In [52]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

In [65]:
data = pd.read_csv('../dataset/data/combined_train.csv')

In [66]:
data = data.dropna(subset=["text", "aspect_term", "polarity"]).reset_index(drop=True)
data = data[(data["text"].astype(str).str.strip() != "") &
            (data["aspect_term"].astype(str).str.strip() != "")].reset_index(drop=True)

data["input_text"] = data["aspect_term"].astype(str) + " [SEP] " + data["text"].astype(str)

In [ ]:
X = data['input_text']
y = data['polarity']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [68]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [69]:
print("X_train type:", type(X_train), X_train.shape)
print("X_test type:", type(X_test), X_test.shape)

X_train type: <class 'scipy.sparse._csr.csr_matrix'> (2176, 5000)
X_test type: <class 'scipy.sparse._csr.csr_matrix'> (545, 5000)


In [70]:
models = {
    'lr': LogisticRegression(max_iter=1000, class_weight="balanced"),
    'svc': LinearSVC(class_weight="balanced", max_iter=5000)
}

results = {}

In [71]:
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")

    results[name] = {"accuracy": acc, "macro_f1": macro_f1, "preds": preds}

    print(f"=== {name.upper()} ===")
    print("Accuracy:", acc)
    print("Macro-F1:", macro_f1)
    print(classification_report(y_test, preds, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print()

=== LR ===
Accuracy: 0.6422018348623854
Macro-F1: 0.48928275507994756
              precision    recall  f1-score   support

    conflict       0.06      0.11      0.08         9
    negative       0.67      0.67      0.67       203
     neutral       0.46      0.53      0.50        94
    positive       0.76      0.68      0.72       239

    accuracy                           0.64       545
   macro avg       0.49      0.50      0.49       545
weighted avg       0.66      0.64      0.65       545

Confusion Matrix:
[[  1   4   1   3]
 [  8 137  26  32]
 [  0  27  50  17]
 [  8  38  31 162]]

=== SVC ===
Accuracy: 0.6972477064220184
Macro-F1: 0.5059936666448366
              precision    recall  f1-score   support

    conflict       0.00      0.00      0.00         9
    negative       0.70      0.73      0.72       203
     neutral       0.52      0.56      0.54        94
    positive       0.78      0.74      0.76       239

    accuracy                           0.70       545
   

In [72]:
best_model_name = max(results, key=lambda k: results[k]["macro_f1"])
best_score = results[best_model_name]["macro_f1"]
print(f"Best model by macro-F1: {best_model_name} ({best_score:.4f})")

Best model by macro-F1: svc (0.5060)
